# Original group notebook (archived as-is). Data paths point to the team's private Google Drive; see the repository README to run the refactored code.

# We have a README file to explain all our main files.

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

d:\anaconda_environment\new_tib\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

games = pd.read_csv('dataset/final_steam_game_data.csv')
user_history = pd.read_csv('dataset/user_game_with_genre.csv')


In [ ]:

def concat_features(row):
    return ' '.join([
        str(row.get('About the game', '')),
        str(row.get('Tags', '')),
        str(row.get('Genres', '')),
        str(row.get('Categories', '')),
        str(row.get('Developers', '')),
        str(row.get('Publishers', ''))
    ])
games['content'] = games.apply(concat_features, axis=1)

In [ ]:
print(games.columns)

Index(['AppID', 'rating', 'positive_ratio', 'user_reviews', 'Name',
       'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price',
       'Discount', 'About the game', 'Supported languages',
       'Full audio languages', 'Windows', 'Mac', 'Linux', 'Metacritic score',
       'Positive', 'Negative', 'Achievements', 'Recommendations',
       'Average playtime forever', 'Average playtime two weeks',
       'Median playtime forever', 'Median playtime two weeks', 'Developers',
       'Publishers', 'Categories', 'Genres', 'Tags', 'content'],
      dtype='object')


In [ ]:
# 3. BERT embedding
model = SentenceTransformer('all-MiniLM-L6-v2')
game_embeddings = model.encode(games['content'].tolist(), show_progress_bar=True)


Batches:   0%|          | 0/1287 [00:00<?, ?it/s]d:\anaconda_environment\new_tib\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Batches: 100%|██████████| 1287/1287 [01:11<00:00, 18.00it/s]


In [ ]:
# 4. Genres multi-tag one-hot coding
def split_and_strip(s):
    if pd.isna(s): return []
    return [x.strip() for x in str(s).replace(';', ',').split(',') if x.strip()]
games['genres_list'] = games['Genres'].apply(split_and_strip)
mlb_genre = MultiLabelBinarizer()
genre_matrix = mlb_genre.fit_transform(games['genres_list'])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

def owners_str_to_num(s):
    if pd.isna(s):
        return 0
    if '-' in str(s):
        parts = str(s).replace(',', '').split('-')
        try:
            low = int(parts[0].strip())
            high = int(parts[1].strip())
            return (low + high) // 2
        except:
            return 0
    else:
        try:
            return int(str(s).replace(',', '').strip())
        except:
            return 0

games['Estimated owners numeric'] = games['Estimated owners'].apply(owners_str_to_num)
games['owners_norm'] = scaler.fit_transform(games[['Estimated owners numeric']])

games['Price'] = pd.to_numeric(games['Price'], errors='coerce').fillna(0)
games['Price_norm'] = scaler.fit_transform(games[['Price']])

games['Metacritic score'] = pd.to_numeric(games['Metacritic score'], errors='coerce').fillna(0)
games['Metacritic_norm'] = scaler.fit_transform(games[['Metacritic score']])


In [ ]:

def get_user_profile(user_id):
    played = user_history[(user_history['user_id']==user_id) & (user_history['is_recommended']==True)]
    if played.empty:
        return None
    indices = games[games['AppID'].isin(played['app_id'])].index
    if len(indices) == 0:
        return None
    hours = played['hours'].values

    user_content_vec = np.average(game_embeddings[indices], axis=0, weights=hours)

    user_genre_vec = np.average(genre_matrix[indices], axis=0, weights=hours)

    user_price = games.iloc[indices]['Price_norm'].mean()

    return user_content_vec, user_genre_vec, user_price

In [ ]:

def recommend_for_user(user_id, alpha=0.7, beta=0.2, gamma=0.1, topN=10):
    profile = get_user_profile(user_id)
    if profile is None:
        print(f"User {user_id} has no positive history.")
        return pd.DataFrame()
    user_content, user_genre, user_price = profile

    content_score = np.dot(game_embeddings, user_content)

    genre_score = genre_matrix @ user_genre

    price_score = 1 - np.abs(games['Price_norm'] - user_price)

    final_score = (alpha * content_score +
                   beta * genre_score +
                   gamma * price_score)

    played = set(user_history[user_history['user_id']==user_id]['app_id'])
    recommend_indices = [i for i in final_score.argsort()[::-1] if games.iloc[i]['AppID'] not in played][:topN]
    return games.iloc[recommend_indices][['AppID', 'Name', 'Genres', 'Price']]

In [ ]:

user_id =0
recs_df = recommend_for_user(user_id, topN=10)
if not recs_df.empty:
    recs_df['user_id'] = user_id
    # recs_df.to_csv(f'recommend_for_user_{user_id}.csv', index=False)

    print(recs_df)
else:
    print(f"User {user_id} There is no available historical data to recommend.")

         AppID                               Name  \
9775    552500            Warhammer: Vermintide 2   
26022   211580                Wanderlust: Rebirth   
10425   981600                           Eternium   
24949   514990                           Outbreak   
7147    218820  Mercenary Kings: Reloaded Edition   
1869   1815530     Dungeon Defenders: Going Rogue   
26516   695360                            Veritex   
2726   2196790                   Jianghu Survivor   
29112   519190                 Next Day: Survival   
38537    65800                  Dungeon Defenders   

                                                  Genres  Price  user_id  
9775                                        Action,Indie  29.99        0  
26022                         Action,Adventure,Indie,RPG   4.99        0  
10425  Action,Adventure,Free to Play,Indie,RPG,Early ...   0.00        0  
24949                         Action,Adventure,Indie,RPG   9.99        0  
7147                          Action,Adve

In [ ]:
# #  Recommended Top-N for all users
# results = []
# for user_id in recs['user_id'].unique():
#     recs_df = recommend_for_user(user_id, topN=10)
#     if not recs_df.empty:
#         recs_df['user_id'] = user_id
#         results.append(recs_df)


# df_results = pd.concat(results, ignore_index=True)
# df_results.to_csv('final_multi_dim_recommend_result.csv', index=False)
# print("Recommendations have been completed and the results are saved in the final_multi_dim_recommend_result.csv")